# Sparkov Credit Card Fraud Detection: M0–M2 Modelling

## 1. Introduction

This notebook implements and evaluates three stages of credit card fraud detection using the Sparkov dataset. The modelling framework progressively addresses key challenges in fraud detection: severe class imbalance, evolving transaction patterns, and delays in fraud-label verification.

The experiments are organised into three stages:

- **M0 – Baseline Models:** Establish reference performance using Logistic Regression, Random Forest, and CatBoost without class-imbalance adjustments.
- **M1 – Cost-Sensitive Learning:** Introduce class weighting to increase the penalty associated with misclassifying fraudulent transactions.
- **M2 – Sliding-Window Retraining:** Investigate whether periodically updating models using recent historical transactions improves robustness to temporal changes, while accounting for a seven-day label verification delay.

Models are evaluated using chronological data partitions to reflect the temporal nature of fraud detection and avoid the optimistic estimates that may arise from random train-test splitting. Precision-Recall AUC (PR-AUC) is the primary evaluation metric due to the low prevalence of fraudulent transactions, supplemented by ROC-AUC, precision, recall, and F1-score.

The final test set is reserved for evaluation after model development.

## 2. Data Loading and Preparation

### 2.1 Load Processed Features

The processed Sparkov dataset is loaded from the feature-engineering pipeline. It contains transaction-level characteristics, temporal features, and historical card behaviour, including rolling-window transaction counts and amounts.

A predefined list of 62 modelling features is loaded separately to exclude transaction identifiers, raw timestamps, target labels, and other non-predictive columns.

The dataset retains chronological training, validation, and test partitions. Their sizes and fraud prevalence are examined to establish the class distribution across evaluation periods.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed")

df = pd.read_pickle(DATA_DIR / "sparkov_features.pkl")

with open(DATA_DIR / "feature_lists.json", "r") as f:
    feature_lists = json.load(f)

feature_cols = feature_lists["sparkov_feature_cols"]

print("Dataset shape:", df.shape)
print("Number of model features:", len(feature_cols))

print("\nChronological splits:")
print(df["sparkov_split"].value_counts())

print("\nFraud rate by split:")
print(df.groupby("sparkov_split")["is_fraud"].mean())

Dataset shape: (1852394, 87)
Number of model features: 62

Chronological splits:
sparkov_split
train         1037340
test           555719
validation     259335
Name: count, dtype: int64

Fraud rate by split:
sparkov_split
test          0.003860
train         0.005753
validation    0.005931
Name: is_fraud, dtype: float64


### 2.2 Prepare Training and Validation Sets

The dataset is partitioned according to the predefined chronological split indicators. The training subset is used for model fitting, while the validation subset is reserved for model comparison and hyperparameter selection.

Predictor variables (`X`) are separated from fraud labels (`y`). Categorical and numerical features are identified to support model-specific preprocessing, and missing values are inspected before model fitting.

The test subset is excluded from model development to prevent information leakage into model selection.

In [2]:
train_mask = df["sparkov_split"] == "train"
val_mask = df["sparkov_split"] == "validation"

X_train = df.loc[train_mask, feature_cols].copy()
y_train = df.loc[train_mask, "is_fraud"].astype(int)

X_val = df.loc[val_mask, feature_cols].copy()
y_val = df.loc[val_mask, "is_fraud"].astype(int)

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_cols = [
    col for col in feature_cols
    if col not in categorical_cols
]

# Release the full dataset after extracting the required subsets
del df
import gc
gc.collect()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Categorical features:", categorical_cols)
print("Numerical features:", len(numerical_cols))

X_train: (1037340, 62)
X_val: (259335, 62)
Categorical features: ['category', 'gender', 'state']
Numerical features: 59


### 2.3 Missing-Value Assessment

Before model training, we examine missing values across the 62 selected predictors to determine whether preprocessing is required. Missing values may arise naturally from historical transaction features, particularly when insufficient prior card activity is available to calculate statistics such as previous transaction amounts or historical variability.

Missing-value counts and proportions are calculated separately for the training and validation sets to identify affected features and assess whether their missingness patterns differ across time. This distinction is important because inappropriate treatment of missing values may distort transaction characteristics or introduce bias into model evaluation.

Rather than immediately removing incomplete observations, we first examine the extent and distribution of missingness. An appropriate preprocessing strategy will subsequently be selected based on feature characteristics and model requirements, with any imputation parameters estimated exclusively from the training data to prevent information leakage.

In [3]:
missing_summary = pd.DataFrame({
    "Train Missing": X_train.isna().sum(),
    "Train Missing (%)": X_train.isna().mean() * 100,
    "Validation Missing": X_val.isna().sum(),
    "Validation Missing (%)": X_val.isna().mean() * 100
})

missing_summary = missing_summary[
    (missing_summary["Train Missing"] > 0) |
    (missing_summary["Validation Missing"] > 0)
].sort_values("Train Missing", ascending=False)

display(missing_summary.round(2))

,Train Missing,Train Missing (%),Validation Missing,Validation Missing (%)
amt_mean_1h,865064,83.39,219064,84.47
amt_mean_24h,87724,8.46,21195,8.17
card_previous_amt_std,1928,0.19,38,0.01
amt_prior_zscore,1928,0.19,38,0.01
amt_mean_7D,1209,0.12,59,0.02
implied_speed_kmh,979,0.09,24,0.01
card_previous_amt_mean,964,0.09,19,0.01
amount_ratio_to_previous,964,0.09,19,0.01
amount_change_from_previous,964,0.09,19,0.01
previous_card_amt,964,0.09,19,0.01


**Missing-Value Assessment: Findings**

Missingness is concentrated in features derived from historical card activity. In particular, `amt_mean_1h` has the highest missingness (83.39% in training and 84.47% in validation), followed by `amt_mean_24h` (8.46% and 8.17%, respectively). Most remaining features exhibit missingness below 0.2%.

These patterns are consistent with structurally undefined historical statistics when insufficient prior transactions are available within the specified observation windows. Consequently, removing incomplete observations would discard a substantial proportion of otherwise valid transactions and potentially introduce selection bias.

We therefore retain all observations and address missing values through model-appropriate preprocessing. For models requiring complete numerical inputs, imputation will be fitted exclusively on the training data and applied consistently to validation data. Historical transaction-count features are retained to provide information about the availability of prior card activity.

### 2.4 Model-Specific Preprocessing

Different classification algorithms impose different requirements on input features. To ensure a fair comparison across baseline models, preprocessing is tailored to each algorithm while maintaining identical training and validation partitions.

For Logistic Regression, categorical variables are one-hot encoded, numerical missing values are imputed using training-set medians, and numerical features are standardised to improve optimisation stability. Random Forest uses categorical encoding and numerical imputation but does not require feature scaling. CatBoost supports categorical predictors directly and can handle missing numerical values.

All learned preprocessing transformations are fitted exclusively on the training partition and subsequently applied to validation data to prevent information leakage. No class weighting or resampling is introduced at this stage, ensuring that M0 represents an unadjusted baseline against which the cost-sensitive M1 models can be compared.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical preprocessing: median imputation and standardisation
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing: imputation and one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing for both feature types
lr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

print("Logistic Regression preprocessing pipeline created.")

Logistic Regression preprocessing pipeline created.


## 3. M0: Baseline Models

M0 establishes reference performance for credit card fraud detection before introducing techniques to address class imbalance or concept drift. Three supervised classification algorithms are considered: Logistic Regression, Random Forest, and CatBoost.

These models represent different levels of complexity, allowing us to assess whether nonlinear relationships and interactions among transaction features improve fraud detection relative to a linear baseline.

All models are trained on the same chronological training partition and evaluated on the validation partition. No class weighting or resampling is applied in M0, preserving a consistent reference for evaluating subsequent improvements in M1 and M2.

### 3.1 Logistic Regression

Logistic Regression serves as an interpretable linear baseline that estimates the probability of a transaction being fraudulent from a weighted combination of its predictors.

Numerical features are median-imputed and standardised, while categorical features are one-hot encoded. These transformations are incorporated into a unified pipeline to ensure consistent preprocessing and prevent validation information from influencing model fitting.

The model uses L2 regularisation and default class weights. Predicted fraud probabilities are evaluated primarily using PR-AUC, with ROC-AUC reported as a complementary threshold-independent metric.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
import time

# Combine preprocessing and classification
lr_model = Pipeline(steps=[
    ("preprocessor", lr_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight=None,
        random_state=42
    ))
])

# Train baseline model
start_time = time.time()

lr_model.fit(X_train, y_train)

training_time = time.time() - start_time

# Predict fraud probabilities on validation data
lr_val_probs = lr_model.predict_proba(X_val)[:, 1]

# Evaluate threshold-independent discrimination
lr_pr_auc = average_precision_score(y_val, lr_val_probs)
lr_roc_auc = roc_auc_score(y_val, lr_val_probs)

print(f"Training time: {training_time:.2f} seconds")
print(f"Validation PR-AUC: {lr_pr_auc:.4f}")
print(f"Validation ROC-AUC: {lr_roc_auc:.4f}")

Training time: 26.33 seconds
Validation PR-AUC: 0.6802
Validation ROC-AUC: 0.9664


**Logistic Regression: Validation Results**

The unweighted Logistic Regression baseline achieved a validation Average Precision (AP) of **0.6802** and ROC-AUC of **0.9664**, with a training time of **26.33 seconds**.

The model demonstrates substantial discriminatory ability despite the severe class imbalance, with AP considerably exceeding the random-ranking reference corresponding to the validation fraud prevalence (0.0059). The high ROC-AUC further indicates that fraudulent transactions generally receive higher predicted risk scores than legitimate transactions.

Nevertheless, these threshold-independent metrics do not directly establish operational detection performance at a specified false-positive rate or alert volume. The results therefore serve as an initial reference for subsequent comparisons with Random Forest, CatBoost, and cost-sensitive learning approaches.

### 3.2 Random Forest

Random Forest is an ensemble learning algorithm that combines multiple decision trees to capture nonlinear relationships and interactions among transaction features. Unlike Logistic Regression, Random Forest does not assume a linear relationship between predictors and the log-odds of fraud, making it suitable for modelling complex transaction behaviours.

For preprocessing, numerical missing values are imputed using training-set medians, while categorical variables are one-hot encoded. Feature standardisation is unnecessary because tree-based models partition observations using feature thresholds rather than distance-based calculations.

To control computational cost and reduce overfitting, the model uses 100 decision trees, a maximum tree depth of 15, and a minimum of 10 samples per leaf. Class weighting remains disabled to preserve the unadjusted M0 baseline.

The model is evaluated on the same chronological validation partition as Logistic Regression, using Average Precision (AP) and ROC-AUC to ensure comparability.


In [7]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest does not require numerical scaling
rf_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

rf_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", rf_numeric_transformer, numerical_cols),
        ("cat", rf_categorical_transformer, categorical_cols)
    ]
)

# Define Random Forest baseline
rf_model = Pipeline(steps=[
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight=None,
        n_jobs=2,
        random_state=42
    ))
])

# Train and evaluate
start_time = time.time()

rf_model.fit(X_train, y_train)

rf_training_time = time.time() - start_time

rf_val_probs = rf_model.predict_proba(X_val)[:, 1]

rf_pr_auc = average_precision_score(y_val, rf_val_probs)
rf_roc_auc = roc_auc_score(y_val, rf_val_probs)

print(f"Training time: {rf_training_time:.2f} seconds")
print(f"Validation PR-AUC: {rf_pr_auc:.4f}")
print(f"Validation ROC-AUC: {rf_roc_auc:.4f}")

Training time: 416.09 seconds
Validation PR-AUC: 0.9589
Validation ROC-AUC: 0.9981


**Random Forest: Validation Results**

The unweighted Random Forest achieved a validation Average Precision (AP) of **0.9589** and ROC-AUC of **0.9981**, outperforming Logistic Regression (AP = 0.6802; ROC-AUC = 0.9664). This suggests that nonlinear relationships and interactions among transaction features may provide substantial predictive value beyond a linear decision boundary.

However, the improvement comes at a higher computational cost, with training requiring **435.28 seconds**, compared with 26.33 seconds for Logistic Regression.

Given the exceptionally high discrimination metrics, further checks are necessary to verify that historical features were constructed using only information available at prediction time and that no target-derived information leaked into the validation data. The results should therefore be treated as provisional until these checks are completed.

In [8]:
importances = rf_model.named_steps["classifier"].feature_importances_

feature_names = rf_model.named_steps[
    "preprocessor"
].get_feature_names_out()

rf_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    .sort_values("Importance", ascending=False)
    .head(15)
)

display(rf_importance)

,Feature,Importance
28,num__amt_x_night,0.109595
46,num__amt_mean_24h,0.090721
0,num__amt,0.080649
14,num__amt_log1p,0.062170
45,num__amt_sum_24h,0.059180
33,num__previous_card_amt,0.047061
17,num__amt_bucket,0.044317
49,num__amt_mean_7D,0.038336
57,num__category_freq,0.036536
35,num__amount_ratio_to_previous,0.035705


**Random Forest: Feature Importance Analysis**

Random Forest feature importance indicates that transaction amounts and historical spending behaviour are influential predictors of fraud. The most important features include `amt_x_night` (0.1096), `amt_mean_24h` (0.0907), and `amt` (0.0806), suggesting that both individual transaction characteristics and recent card activity contribute to the model's predictions.

Historical spending features, including `amt_sum_24h`, `previous_card_amt`, and `amount_ratio_to_previous`, also rank highly. These findings are consistent with the potential value of behavioural deviations for fraud detection.

However, impurity-based importance does not establish causality or rule out information leakage. Given the exceptionally high validation performance, further verification of historical feature construction and temporal availability is necessary before interpreting the results as evidence of generalisable predictive performance.

### 3.2.1 Validation of Historical Feature Construction

Given the high validation performance of Random Forest, an independent consistency check is conducted to assess potential information leakage in the engineered historical transaction features.

For a sample of cards, the stored 24-hour transaction counts, sums, and means are compared against values independently reconstructed from raw transaction timestamps and amounts. Only transactions belonging to the same card and occurring strictly before the current transaction within the preceding 24 hours are included.

This check also assesses whether the engineered rolling statistics remain correctly aligned with their corresponding transactions after card-level sorting and feature assignment.

In [9]:
# Independent leakage check for 24-hour historical features
from pathlib import Path
import numpy as np
import pandas as pd

# Load only the columns needed for validation
check_cols = [
    "cc_num",
    "trans_date_trans_time",
    "amt",
    "txn_count_24h",
    "amt_sum_24h",
    "amt_mean_24h"
]

check_df = pd.read_pickle(
    Path("../data/processed/sparkov_features.pkl")
)[check_cols].copy()

check_df["trans_date_trans_time"] = pd.to_datetime(
    check_df["trans_date_trans_time"]
)

# Select 10 reproducible sample cards
sample_cards = (
    check_df["cc_num"]
    .drop_duplicates()
    .sample(n=10, random_state=42)
)

sample = check_df[
    check_df["cc_num"].isin(sample_cards)
].copy()

sample = sample.sort_values(
    ["cc_num", "trans_date_trans_time"],
    kind="stable"
)

# Independently reconstruct prior 24-hour history
results = []

for card, group in sample.groupby("cc_num"):
    group = group.sort_values("trans_date_trans_time")

    times = group["trans_date_trans_time"].to_numpy()
    amounts = group["amt"].to_numpy()

    for i in range(len(group)):
        current_time = times[i]
        window_start = current_time - np.timedelta64(24, "h")

        prior_mask = (
            (times < current_time) &
            (times >= window_start)
        )

        prior_amounts = amounts[prior_mask]

        expected_count = len(prior_amounts)
        expected_sum = prior_amounts.sum()
        expected_mean = (
            prior_amounts.mean()
            if expected_count > 0 else np.nan
        )

        row = group.iloc[i]

        results.append({
            "count_match": np.isclose(
                row["txn_count_24h"], expected_count
            ),
            "sum_match": np.isclose(
                row["amt_sum_24h"], expected_sum,
                rtol=1e-5, atol=1e-5
            ),
            "mean_match": np.isclose(
                row["amt_mean_24h"], expected_mean,
                rtol=1e-5, atol=1e-5,
                equal_nan=True
            )
        })

results_df = pd.DataFrame(results)

print("Transactions checked:", len(results_df))
print("Count mismatches:", (~results_df["count_match"]).sum())
print("Sum mismatches:", (~results_df["sum_match"]).sum())
print("Mean mismatches:", (~results_df["mean_match"]).sum())

Transactions checked: 19776
Count mismatches: 0
Sum mismatches: 0
Mean mismatches: 0


**Historical Feature Validation: Results**

An independent consistency check was performed on 19,776 transactions from 10 randomly sampled cards. The stored 24-hour transaction counts, sums, and means were compared against values reconstructed using only transactions occurring strictly before each transaction within the preceding 24 hours.

No discrepancies were identified across the three features. These findings support the correctness of the 24-hour rolling-window calculations and their alignment with individual transactions in the sampled data.

Although this check does not eliminate all possible sources of information leakage, it provides evidence that the exceptionally high Random Forest validation performance is not attributable to incorrect construction of these particular historical features.

### 3.3 CatBoost

CatBoost is a gradient-boosted decision tree algorithm that sequentially constructs trees to correct errors made by preceding iterations. Unlike Random Forest, which aggregates independently constructed trees, CatBoost uses boosting to progressively improve predictive performance.

CatBoost supports categorical predictors directly, eliminating the need for one-hot encoding, and handles missing numerical values natively. Consequently, numerical features are retained without scaling or imputation, while missing categorical values are replaced with an explicit category.

For the M0 baseline, CatBoost is configured with 300 boosting iterations, a learning rate of 0.05, and a maximum tree depth of 6. Class weighting is disabled to maintain comparability with the unweighted Logistic Regression and Random Forest baselines.

The model is trained on the same chronological training partition and evaluated using Average Precision (AP) and ROC-AUC on the validation partition.

In [10]:
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
import time

# Prepare CatBoost-specific inputs
# Avoid copying the full numerical dataset unnecessarily
X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

# CatBoost requires categorical columns to contain strings
for col in categorical_cols:
    X_train_cb[col] = X_train_cb[col].fillna("Missing").astype(str)
    X_val_cb[col] = X_val_cb[col].fillna("Missing").astype(str)

# Define unweighted CatBoost baseline
cb_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    cat_features=categorical_cols,
    auto_class_weights=None,
    thread_count=2,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

# Train model
start_time = time.time()

cb_model.fit(X_train_cb, y_train)

cb_training_time = time.time() - start_time

# Validation predictions
cb_val_probs = cb_model.predict_proba(X_val_cb)[:, 1]

cb_pr_auc = average_precision_score(y_val, cb_val_probs)
cb_roc_auc = roc_auc_score(y_val, cb_val_probs)

print(f"Training time: {cb_training_time:.2f} seconds")
print(f"Validation PR-AUC: {cb_pr_auc:.4f}")
print(f"Validation ROC-AUC: {cb_roc_auc:.4f}")

Training time: 283.82 seconds
Validation PR-AUC: 0.9791
Validation ROC-AUC: 0.9993


**CatBoost: Validation Results**

The unweighted CatBoost baseline achieved a validation Average Precision (AP) of **0.9791** and ROC-AUC of **0.9993**, outperforming both Logistic Regression and Random Forest. Its training time of **283.82 seconds** was lower than Random Forest's 435.28 seconds, although substantially higher than Logistic Regression's 26.33 seconds.

These findings suggest that gradient-boosted decision trees are effective at capturing complex relationships among transaction characteristics and historical spending behaviour. Nevertheless, the exceptionally high validation performance warrants continued attention to temporal validity and potential information leakage before drawing conclusions about deployment performance.

### 3.4 Baseline Model Comparison

In [11]:
m0_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "PR-AUC (AP)": lr_pr_auc,
        "ROC-AUC": lr_roc_auc,
        "Training Time (s)": training_time
    },
    {
        "Model": "Random Forest",
        "PR-AUC (AP)": rf_pr_auc,
        "ROC-AUC": rf_roc_auc,
        "Training Time (s)": rf_training_time
    },
    {
        "Model": "CatBoost",
        "PR-AUC (AP)": cb_pr_auc,
        "ROC-AUC": cb_roc_auc,
        "Training Time (s)": cb_training_time
    }
])

display(
    m0_results
    .sort_values("PR-AUC (AP)", ascending=False)
    .round(4)
)

,Model,PR-AUC (AP),ROC-AUC,Training Time (s)
2,CatBoost,0.9791,0.9993,283.8226
1,Random Forest,0.9589,0.9981,416.0879
0,Logistic Regression,0.6802,0.9664,26.3307


**M0 Baseline Comparison: Discussion**

Across the three baseline models, CatBoost achieved the strongest validation performance (AP = 0.9791; ROC-AUC = 0.9993), followed by Random Forest (AP = 0.9589; ROC-AUC = 0.9981) and Logistic Regression (AP = 0.6802; ROC-AUC = 0.9664).

The substantial improvement from Logistic Regression to the tree-based ensembles suggests that nonlinear relationships and feature interactions are important for identifying fraudulent transactions. CatBoost also demonstrated a more favourable balance between predictive performance and computational cost than Random Forest under the selected baseline configurations.

These results establish reference performance for subsequent experiments addressing class imbalance and temporal adaptation. However, the comparisons reflect the selected baseline configurations rather than exhaustive hyperparameter optimisation, and the unusually high validation scores require continued scrutiny of feature construction and temporal evaluation assumptions.